# Road Accident Severity Analysis for U.S. Data with a Kenya Application Context

**Source dataset:** U.S. road accidents, 2016â€“2023  
**Working dataset in this notebook:** first 50,000 rows from the source CSV  
**Application frame:** Kenyan road-safety planning and evaluation

The original U.S. Accidents file contains approximately 7.7 million records. Because of local memory constraints, this notebook uses a 50,000-row working sample rather than loading the entire dataset into memory. All EDA, statistical analysis, feature engineering, model training, testing, and evaluation in this notebook are based on that 50,000-row sample.

The Kenya context is used to discuss feasibility, data requirements, and transferability. It is not used to claim that U.S. patterns are Kenyan patterns.

The project follows the CRISP-DM sequence: Business Understanding, Data Understanding, Data Preparation, Exploratory Data Analysis, Modeling, Evaluation, Kenya Application / Business Evaluation, and Conclusion.

### Analytical questions
1. Which conditions are associated with more severe accidents?
2. Are there identifiable temporal patterns?
3. What environmental and road characteristics appear alongside severe accidents?
4. Can accident severity be predicted from information available at the time of an incident?
5. What information would a Kenyan road-safety system need?
6. Which U.S. variables would require Kenyan equivalents, and what additional Kenyan data would be required before local deployment?



## 6. Evaluation

The primary metric is recall for the high-severity class because missing a high-severity accident is a more serious operational problem than creating a false alert. Precision and F1 are still reported so the alert burden remains visible.

### Selected model
The model with the highest recall on the chronological holdout set is chosen for detailed review. This is a deliberate choice tied to the operational objective, not a claim that any model is automatically valid for Kenya.


In [ ]:

selected_model_name = results_df.iloc[0]['model']
selected_model = trained_models[selected_model_name]
selected_pred = selected_model.predict(X_test)
selected_prob = selected_model.predict_proba(X_test)[:, 1]

print(f'Selected model: {selected_model_name}')
print(f'High-severity recall: {results_df.iloc[0]["recall"]:.3f}')
print(f'Precision: {results_df.iloc[0]["precision"]:.3f}')
print(f'F1: {results_df.iloc[0]["f1"]:.3f}')
print(f'ROC-AUC: {roc_auc_score(y_test, selected_prob):.3f}')
print(f'PR-AUC: {average_precision_score(y_test, selected_prob):.3f}')
print('\nClassification report:')
print(classification_report(y_test, selected_pred, target_names=['Lower severity', 'High severity']))

cm = confusion_matrix(y_test, selected_pred)
print('\nConfusion matrix:')
print(cm)

plt.figure(figsize=(6, 5))
cm_display = plt.imshow(cm, cmap='Blues')
plt.xticks([0, 1], ['Lower', 'High'])
plt.yticks([0, 1], ['Lower', 'High'])
plt.title(f'Confusion matrix for {selected_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.colorbar(cm_display)
plt.show()



### Interpretation of the evaluation
The model should be evaluated by the business question it is meant to support. If it catches many high-severity cases but produces many false alarms, it may still be useful as a screening aid if reviewed by staff. If it misses many serious events, it is not suitable as a no-human-check system.

This is not a claim that the model is valid in Kenya. It is a U.S.-trained evaluation of a U.S. dataset.
